In [1]:
import os
import sys

sys.path.append(os.getcwd())

In [2]:
from utils.data_loader import load_target_text
from taxonomy.taxonomy import Taxonomy
from models.tagrec import TagRec
from llm.llmJudge import LLMJudge
from ppi.ppi import LPPI
import pandas as pd

tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
questions = load_target_text("data\\d1\\train.csv", target_column = "eng", top_n=10, keep_columns=[])
taxonomy = Taxonomy("taxonomy\\PCv3.txt")

tagger = TagRec()
tagger.set_taxonomy(taxonomy)
tagger.load()

llm_judge = LLMJudge(model="openai/gpt-oss-120b")

lppi = LPPI()
lppi_gold = pd.read_pickle("data\\d1\\lppi_gold1.pkl")
lppi.fit(lppi_gold, score_cols=["Subject_validity", "Chapter_validity", "Topic_validity"])

✅ TagRec loaded successfully.
Loading embedding model 'all-MiniLM-L6-v2' on cuda...
Fitting LPPI...
Embedding gold text...


Batches: 100%|██████████| 58/58 [00:03<00:00, 17.03it/s]

LPPI Fit complete.


In [6]:
tagrec_predictions = tagger.predict(questions, top_k=3)
tagrec_predictions

,target,predicted_taxonomy
0,Photodiode is a device\nA. Which is always ope...,TaxonomyNode(Physics >> Dual Nature of Radiati...
1,Photodiode is a device\nA. Which is always ope...,TaxonomyNode(Physics >> Dual Nature of Radiati...
2,Photodiode is a device\nA. Which is always ope...,TaxonomyNode(Physics >> Dual Nature of Radiati...
3,When water is heated from \( 0^{\circ} \mathrm...,TaxonomyNode(Physics >> Thermal Properties of ...
4,When water is heated from \( 0^{\circ} \mathrm...,TaxonomyNode(Physics >> Thermal Properties of ...
5,When water is heated from \( 0^{\circ} \mathrm...,TaxonomyNode(Physics >> Thermodynamics >> Spec...
6,Potentiometer measures the potential differenc...,TaxonomyNode(Physics >> Current Electricity >>...
7,Potentiometer measures the potential differenc...,TaxonomyNode(Physics >> Current Electricity >>...
8,Potentiometer measures the potential differenc...,TaxonomyNode(Physics >> Current Electricity >>...
9,"In an isosceles trapezium, the length of one o...",TaxonomyNode(Physics >> Nuclei >> Atomic masse...


In [7]:
tagrec_predictions_score_llm = llm_judge.judge_taxonomy_predictions(tagrec_predictions)
tagrec_predictions_score_llm

Judging row 0...
Judging row 1...
Judging row 2...
Judging row 3...
Judging row 4...
Judging row 5...
Judging row 6...
Judging row 7...
Judging row 8...
Judging row 9...
Judging row 10...
Judging row 11...
Judging row 12...
Judging row 13...
Judging row 14...
Judging row 15...
Judging row 16...
Judging row 17...
Judging row 18...
Judging row 19...
Judging row 20...
Judging row 21...
Judging row 22...
Judging row 23...
Judging row 24...
Judging row 25...
Judging row 26...
Judging row 27...
Judging row 28...
Judging row 29...


,target,predicted_taxonomy,Subject_validity_llm,Chapter_validity_llm,Topic_validity_llm
0,Photodiode is a device\nA. Which is always ope...,TaxonomyNode(Physics >> Dual Nature of Radiati...,0.95,0.62,0.21
1,Photodiode is a device\nA. Which is always ope...,TaxonomyNode(Physics >> Dual Nature of Radiati...,0.96,0.65,0.30
2,Photodiode is a device\nA. Which is always ope...,TaxonomyNode(Physics >> Dual Nature of Radiati...,0.99,0.78,0.72
3,When water is heated from \( 0^{\circ} \mathrm...,TaxonomyNode(Physics >> Thermal Properties of ...,0.99,0.92,0.10
4,When water is heated from \( 0^{\circ} \mathrm...,TaxonomyNode(Physics >> Thermal Properties of ...,0.92,0.88,0.15
5,When water is heated from \( 0^{\circ} \mathrm...,TaxonomyNode(Physics >> Thermodynamics >> Spec...,0.97,0.88,0.12
6,Potentiometer measures the potential differenc...,TaxonomyNode(Physics >> Current Electricity >>...,0.99,0.94,0.32
7,Potentiometer measures the potential differenc...,TaxonomyNode(Physics >> Current Electricity >>...,0.99,0.94,0.18
8,Potentiometer measures the potential differenc...,TaxonomyNode(Physics >> Current Electricity >>...,0.99,0.97,0.98
9,"In an isosceles trapezium, the length of one o...",TaxonomyNode(Physics >> Nuclei >> Atomic masse...,0.02,0.01,0.01


In [8]:
lppi.calibrate(tagrec_predictions_score_llm)

Embedding unlabeled text...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.65it/s]

Finding nearest gold neighbors...
Applying local rectification...


,target,predicted_taxonomy,Subject_validity_llm,Chapter_validity_llm,Topic_validity_llm,Subject_validity_rectified,Subject_validity_ci_size,Chapter_validity_rectified,Chapter_validity_ci_size,Topic_validity_rectified,Topic_validity_ci_size
0,Photodiode is a device\nA. Which is always ope...,TaxonomyNode(Physics >> Dual Nature of Radiati...,0.95,0.62,0.21,0.981,0.021181,0.303,0.244923,0.069,0.127408
1,Photodiode is a device\nA. Which is always ope...,TaxonomyNode(Physics >> Dual Nature of Radiati...,0.96,0.65,0.30,0.991,0.021181,0.333,0.244923,0.159,0.127408
2,Photodiode is a device\nA. Which is always ope...,TaxonomyNode(Physics >> Dual Nature of Radiati...,0.99,0.78,0.72,1.021,0.021181,0.463,0.244923,0.579,0.127408
3,When water is heated from \( 0^{\circ} \mathrm...,TaxonomyNode(Physics >> Thermal Properties of ...,0.99,0.92,0.10,1.019,0.012367,0.625,0.284434,-0.271,0.203355
4,When water is heated from \( 0^{\circ} \mathrm...,TaxonomyNode(Physics >> Thermal Properties of ...,0.92,0.88,0.15,0.949,0.012367,0.585,0.284434,-0.221,0.203355
5,When water is heated from \( 0^{\circ} \mathrm...,TaxonomyNode(Physics >> Thermodynamics >> Spec...,0.97,0.88,0.12,0.999,0.012367,0.585,0.284434,-0.251,0.203355
6,Potentiometer measures the potential differenc...,TaxonomyNode(Physics >> Current Electricity >>...,0.99,0.94,0.32,1.001,0.002262,1.002,0.027355,0.001,0.190954
7,Potentiometer measures the potential differenc...,TaxonomyNode(Physics >> Current Electricity >>...,0.99,0.94,0.18,1.001,0.002262,1.002,0.027355,-0.139,0.190954
8,Potentiometer measures the potential differenc...,TaxonomyNode(Physics >> Current Electricity >>...,0.99,0.97,0.98,1.001,0.002262,1.032,0.027355,0.661,0.190954
9,"In an isosceles trapezium, the length of one o...",TaxonomyNode(Physics >> Nuclei >> Atomic masse...,0.02,0.01,0.01,0.055,0.012275,0.065,0.051502,-0.288,0.275715
